In [0]:
# Databricks notebook source
dbutils.widgets.text("bronze_catalog", "bronze_germantelekom")
dbutils.widgets.text("silver_catalog", "silver_germantelekom")
dbutils.widgets.text("gold_catalog", "gold_germantelekom")
dbutils.widgets.text("schema", "customer")

bronze_catalog = dbutils.widgets.get("bronze_catalog")
silver_catalog = dbutils.widgets.get("silver_catalog")
gold_catalog = dbutils.widgets.get("gold_catalog")
schema = dbutils.widgets.get("schema")

bronze_tbl = f"{bronze_catalog}.{schema}.customer_data"
silver_tbl = f"{silver_catalog}.{schema}.customer_data"
quarantine_tbl = f"{silver_catalog}.{schema}.customer_data_quarantine"
run_log_tbl = f"{silver_catalog}.{schema}.dq_run_log"
gold_summary_tbl = f"{gold_catalog}.{schema}.customer_summary_by_pincode"
gold_current_vw = f"{gold_catalog}.{schema}.customer_current"

# COMMAND ----------

# SCHEMAS: created inside the existing catalogs
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_catalog}.{schema} COMMENT 'Raw customer data as received from CSV'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}.{schema} COMMENT 'Cleaned customer data with SCD2 history and quarantine'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{schema} COMMENT 'Business-ready customer views and aggregates'")

# COMMAND ----------

# BRONZE: raw, all strings
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_tbl} (
  customer_id        STRING,
  customer_name      STRING,
  pincode            STRING,
  installed_product  STRING,
  speed              STRING,
  segment            STRING,
  _corrupt_record    STRING,
  source_file_name   STRING,
  load_timestamp     TIMESTAMP
) USING DELTA
""")

# COMMAND ----------

# SILVER: cleaned, SCD2 history
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_tbl} (
  customer_id           STRING,
  customer_name         STRING,
  pincode               STRING,
  installed_product     STRING,
  speed_mbps            INT,
  segment               STRING,
  effective_start_date  TIMESTAMP,
  effective_end_date    TIMESTAMP,
  is_current            BOOLEAN
) USING DELTA
""")

# SILVER: quarantine
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {quarantine_tbl} (
  customer_id        STRING,
  customer_name      STRING,
  pincode            STRING,
  installed_product  STRING,
  speed              STRING,
  segment            STRING,
  reject_reason      STRING,
  corrupt_record     STRING,
  source_file_name   STRING,
  load_timestamp     TIMESTAMP
) USING DELTA
""")

# SILVER: run log
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {run_log_tbl} (
  run_id            STRING,
  source_file_name  STRING,
  total_rows        BIGINT,
  valid_rows        BIGINT,
  quarantined_rows  BIGINT,
  run_timestamp     TIMESTAMP
) USING DELTA
""")

# COMMAND ----------

# GOLD: summary table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {gold_summary_tbl} (
  pincode            STRING,
  installed_product  STRING,
  speed_mbps         INT,
  segment            STRING,
  customer_count     BIGINT,
  snapshot_date      DATE
) USING DELTA
""")

# GOLD: current snapshot view
spark.sql(f"""
CREATE OR REPLACE VIEW {gold_current_vw} AS
SELECT customer_id, customer_name, pincode, installed_product,
       speed_mbps, segment, effective_start_date AS since_date
FROM {silver_tbl}
WHERE is_current = true
""")

# COMMAND ----------

# Verify
for cat in [bronze_catalog, silver_catalog, gold_catalog]:
    display(spark.sql(f"SHOW TABLES IN {cat}.{schema}"))
